In [ ]:
import pandas as pd
import re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

# Fungsi untuk memproses teks ulasan
def process_reviews(raw_text):
    reviews = re.split(r'\n\n+', raw_text.strip())
    
    structured_data = []
    
    sia = SentimentIntensityAnalyzer()
    
    for review in reviews:
        if len(review.strip()) == 0:
            continue
            
        words = len(review.split())
        sentiment_score = sia.polarity_scores(review)
        
        has_cheater_mention = 1 if re.search(r'cheat(er|ing|s)?', review, re.IGNORECASE) else 0
        has_server_mention = 1 if re.search(r'server', review, re.IGNORECASE) else 0
        has_matchmaking_mention = 1 if re.search(r'match(making)?', review, re.IGNORECASE) else 0
        has_money_mention = 1 if re.search(r'money|cash|expensive|price', review, re.IGNORECASE) else 0
        
        hours_match = re.search(r'(\d+)[+]?\s*(?:hours|hrs)', review, re.IGNORECASE)
        hours_played = int(hours_match.group(1)) if hours_match else None
        
        # Tambahkan ke daftar data terstruktur
        structured_data.append({
            'review_text': review,
            'word_count': words,
            'sentiment_compound': sentiment_score['compound'],
            'sentiment_pos': sentiment_score['pos'],
            'sentiment_neg': sentiment_score['neg'],
            'sentiment_neu': sentiment_score['neu'],
            'mentions_cheaters': has_cheater_mention,
            'mentions_servers': has_server_mention,
            'mentions_matchmaking': has_matchmaking_mention,
            'mentions_money': has_money_mention,
            'hours_played': hours_played
        })
    
    # Konversi ke DataFrame
    df = pd.DataFrame(structured_data)
    
    df['sentiment'] = df['sentiment_compound'].apply(
        lambda score: 'positive' if score >= 0.05 else ('negative' if score <= -0.05 else 'neutral')
    )
    
    return df

def transform_review_data(raw_text):
    # Proses ulasan
    reviews_df = process_reviews(raw_text)
    
    # Hitung statistik
    stats = {
        'total_reviews': len(reviews_df),
        'positive_reviews': sum(reviews_df['sentiment'] == 'positive'),
        'negative_reviews': sum(reviews_df['sentiment'] == 'negative'),
        'neutral_reviews': sum(reviews_df['sentiment'] == 'neutral'),
        'avg_word_count': reviews_df['word_count'].mean(),
        'cheater_mentions': reviews_df['mentions_cheaters'].sum(),
        'server_mentions': reviews_df['mentions_servers'].sum(),
        'matchmaking_mentions': reviews_df['mentions_matchmaking'].sum(),
        'money_mentions': reviews_df['mentions_money'].sum()
    }
    
    return reviews_df, stats


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Voire\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
with open('apex_legends_reviews_only.txt', 'r', encoding='utf-8') as file:
    raw_text = file.read()

reviews_df, stats = transform_review_data(raw_text)

reviews_df.to_csv('apex_reviews_structured.csv', index=False)

print("=== Statistik Ulasan ===")
for key, value in stats.items():
    print(f"{key}: {value}")

=== Statistik Ulasan ===
total_reviews: 199
positive_reviews: 96
negative_reviews: 90
neutral_reviews: 13
avg_word_count: 46.91959798994975
cheater_mentions: 37
server_mentions: 15
matchmaking_mentions: 28
money_mentions: 31
